# Experiment: Benefit of Reflection in Meeting-to-Task Agent

This notebook demonstrates the impact of the **Reflection** step in the AI agent's workflow. We compare two scenarios:
1.  **Baseline (No Reflection)**: The agent performs STT Analysis and stops.
2.  **Full Workflow (With Reflection)**: The agent performs Analysis -> Reflection -> Refinement (if needed).

We visualize the differences in the generated summary and action items.

In [1]:
import sys
import os
import json
import pandas as pd
from IPython.display import display, Markdown

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.append(project_root)
    # Also add ai_service to path for internal absolute imports in models modules
    sys.path.append(os.path.join(project_root, 'ai_service'))

from ai_service.src.agents.meeting_to_task.agent import MeetingToTaskAgent
from ai_service.src.agents.meeting_to_task.schemas import AgentState
from dotenv import load_dotenv

# Load environment variables
load_dotenv(os.path.join(project_root, '.env'))

True

In [2]:
# Initialize Agent
agent = MeetingToTaskAgent()

e:\Individual\Projects\ProMeet\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


In [ ]:
audio_path = "E:\\Individual\\Projects\\ProMeet\\server\\static\\recordings\\69b93cdb-6094-40a7-97da-269fbd262f24.webm"
meeting_metadata = {
    "title": "Aentic Application",
    "description": "Building agentic ai application",
    "project_id": "484a97a2-8fbf-4da3-a42f-ed89bf3d73db",
    "author_id": "52de8e7c-40d3-488f-8888-85d9aaa005eb",
    "participants": [
        {"id": "52de8e7c-40d3-488f-8888-85d9aaa005eb", "name": "Hùng", "email": "nguyenquangphuoc9614@gmail.com"},
        {"id": "754f0dad-3d6d-47d8-bdf7-f9d30496f2f7", "name": "Tuấn",
         "email": "nguyenquangphuoc962004@gmail.com"},
        {"id": "b05deb68-3629-4c68-adb4-8d7dba8034e7", "name": "Lan", "email": "chauthanhdat174@gmail.com"},
        {"id": "dfb297da-7b37-4650-bbde-fb1adaff59be", "name": "Nam", "email": "zzvandatzz9@gmail.com"}
    ]
}

sample_transcript = """
Hùng: Ok mọi người, vào họp nhanh nhé. Hôm nay chúng ta chốt mấy đầu việc cho cái Agentic AI app.
Tuấn: Vâng em nghe đây.
Hùng: Đầu tiên là cái module Authentication. Tuấn, em lo phần này nhé? Anh cần tích hợp Google Login.
Tuấn: Ơ anh ơi, em đang dở cái cổng thanh toán VNPay rồi, chắc không kịp đâu.
Lan: Đúng rồi đấy Hùng, Tuấn đang overload. Để Nam làm cái Auth đi.
Hùng: Ừ hợp lý. Thế Nam, em nhận cái Auth nhé? Tích hợp Google với Github login luôn.
Nam: Ok anh, nhưng em chưa làm Github bao giờ.
Hùng: Không sao, cứ làm đi, rồi Tuấn sẽ review code cho em.
Tuấn: Ok, vậy chốt là em review code Auth của Nam nhé.
Hùng: Tiếp theo, Lan, cái giao diện Dashboard bao giờ xong?
Lan: Chắc thứ 6 tuần này em xong ạ.
Nam: Thứ 6 là ngày 12 ấy hả chị?
Lan: À nhầm, tuần này em bận đám cưới. Phải thứ 6 tuần sau, ngày 19/01 nhé.
Hùng: Ok, deadline 19/01. Ghi chú vào là ưu tiên cao nhé, thiếu dashboard là không demo được đâu.
Tuấn: Còn cái vụ Database Migration sang PostgreSQL thì sao anh?
Hùng: À cái đấy, hôm qua anh bàn với sếp rồi. MVP chưa cần đâu, cứ dùng SQLite cho nhanh.
Tuấn: Thế là thôi không làm Migration nữa đúng không?
Hùng: Đúng rồi, hủy task đấy đi.
Hùng: À còn một việc cuối, quan trọng này. Tất cả mọi người nhớ update lại API Key mới trong file .env nhé, cái cũ hết hạn rồi. Deadline là cuối ngày hôm nay.
"""
thread_id = 'demo_test'

## Scenario 1: No Reflection (Analysis Only)
We manually execute the `analysis` node.

In [8]:

import json

agent = MeetingToTaskAgent()

initial_state = {
    'audio_file_path': audio_path,
    'meeting_metadata': meeting_metadata,
    'transcript': sample_transcript,
    'max_revisions': 0,
    'revision_count': 0,
}

thread_config = {'configurable': {'thread_id': "4123"}}

print("🚀 Running Agent and capturing steps...")
history = []
draft_actions = []

# Storage for Scenario 1
s1_summary = ""
s1_actions = []

# 2. Stream output from the graph
for event in agent.graph.stream(initial_state, thread_config):
    for node_name, output in event.items():
        print(f"  📍 Node '{node_name}' completed.")
        
        # Save snapshot
        step_data = {"node": node_name, "output": output}
        history.append(step_data)
        
        # --- Live Inspection ---
        if node_name == 'analysis':
            draft_actions = output.get('action_items', [])
            s1_summary = output.get('summary', "")
            s1_actions = draft_actions
            print(f"     📝 DRAFT GENERATED ({len(draft_actions)} items):")
            # PRINT FULL JSON
            print(json.dumps(draft_actions, indent=2, ensure_ascii=False))
                
        elif node_name == 'reflection':
            print(f"     🤔 CRITIQUE: {output.get('critique')}")
            print(f"     👉 DECISION: {output.get('reflect_decision')}")
            
        elif node_name == 'refinement':
            refined_actions = output.get('action_items', [])
            print(f"     ✨ REFINED OUTPUT ({len(refined_actions)} items):")
            # PRINT FULL JSON
            print(json.dumps(refined_actions, indent=2, ensure_ascii=False))
            
            # Quick Diff
            if len(refined_actions) != len(draft_actions):
                 print(f"     ⚠️ Count changed: {len(draft_actions)} -> {len(refined_actions)}")

print("\n✅ Workflow Finished")

# 3. Final Critique Summary
print("\n--- Final Reflection Report ---")
for step in history:
    if step['node'] == 'reflection':
        print(f"Critique: {step['output']['critique']}")

🚀 Running Agent and capturing steps...
  📍 Node 'stt' completed.
  📍 Node 'analysis' completed.
     📝 DRAFT GENERATED (3 items):
[
  {
    "title": "Hoàn thành module Authentication",
    "description": "Tích hợp Google và Github login",
    "assignee": "Nam",
    "priority": "High",
    "due_date": "2024-01-19",
    "status": "To Do",
    "tags": null
  },
  {
    "title": "Thiết kế Dashboard",
    "description": "Hoàn thành giao diện Dashboard",
    "assignee": "Lan",
    "priority": "High",
    "due_date": "2024-01-19",
    "status": "To Do",
    "tags": null
  },
  {
    "title": "Cập nhật API Key",
    "description": "Update API Key mới trong file .env",
    "assignee": "Unassigned",
    "priority": "Urgent",
    "due_date": "2024-01-08",
    "status": "To Do",
    "tags": null
  }
]
  📍 Node 'reflection' completed.
     🤔 CRITIQUE: - Completeness: Action Items còn thiếu việc Tuấn review code cho Nam. 
- Accuracy: 
  + Assignee của "Cập nhật API Key" chưa đúng, cần phải là tất cả

## Scenario 2: With Reflection
We run the full graph which includes Reflection and Refinement.

In [9]:
import json

agent = MeetingToTaskAgent()

initial_state = {
    'audio_file_path': audio_path,
    'meeting_metadata': meeting_metadata,
    'transcript': sample_transcript,
    'max_revisions': 2,
    'revision_count': 0,
}

thread_config = {'configurable': {'thread_id': "231"}}

print("🚀 Running Agent and capturing steps...")
history_2 = []
draft_actions_2 = []

# Storage for Scenario 2
s2_summary = ""
s2_actions = []
critiques_log = []

# 2. Stream output from the graph
for event in agent.graph.stream(initial_state, thread_config):
    for node_name, output in event.items():
        print(f"  📍 Node '{node_name}' completed.")
        
        # Save snapshot
        step_data = {"node": node_name, "output": output}
        history_2.append(step_data)
        
        # --- Live Inspection ---
        if node_name == 'analysis':
            draft_actions_2 = output.get('action_items', [])
            print(f"     📝 DRAFT GENERATED ({len(draft_actions_2)} items):")
            # PRINT FULL JSON
            print(json.dumps(draft_actions_2, indent=2, ensure_ascii=False))
                
        elif node_name == 'reflection':
            critique_text = output.get('critique')
            decision = output.get('reflect_decision')
            critiques_log.append(f"Critique: {critique_text} (Decision: {decision})")
            print(f"     🤔 CRITIQUE: {critique_text}")
            print(f"     👉 DECISION: {decision}")
            
        elif node_name == 'refinement':
            refined_actions = output.get('action_items', [])
            
            # Update final results whenever we refine
            s2_summary = output.get('summary', "")
            s2_actions = refined_actions
            
            print(f"     ✨ REFINED OUTPUT ({len(refined_actions)} items):")
            # PRINT FULL JSON
            print(json.dumps(refined_actions, indent=2, ensure_ascii=False))
            
            # Quick Diff
            if len(refined_actions) != len(draft_actions_2):
                 print(f"     ⚠️ Count changed: {len(draft_actions_2)} -> {len(refined_actions)}")

print("\n✅ Workflow Finished")

# 3. Final Critique Summary
print("\n--- Final Reflection Report ---")
for step in history_2:
    if step['node'] == 'reflection':
        print(f"Critique: {step['output']['critique']}")

🚀 Running Agent and capturing steps...
  📍 Node 'stt' completed.
  📍 Node 'analysis' completed.
     📝 DRAFT GENERATED (3 items):
[
  {
    "title": "Hoàn thành module Authentication",
    "description": "Tích hợp Google và Github login",
    "assignee": "Nam",
    "priority": "High",
    "due_date": "2024-01-19",
    "status": "To Do",
    "tags": null
  },
  {
    "title": "Thiết kế Dashboard",
    "description": "Hoàn thành giao diện Dashboard",
    "assignee": "Lan",
    "priority": "High",
    "due_date": "2024-01-19",
    "status": "To Do",
    "tags": null
  },
  {
    "title": "Cập nhật API Key",
    "description": "Update API Key mới trong file .env",
    "assignee": "Unassigned",
    "priority": "Urgent",
    "due_date": "2024-01-08",
    "status": "To Do",
    "tags": null
  }
]
  📍 Node 'reflection' completed.
     🤔 CRITIQUE: - Completeness: Action Items còn thiếu việc Tuấn review code cho Nam. 
- Accuracy: 
  + Logic: Đúng, Nam nhận Auth sau khi Tuấn từ chối. 
  + Details

In [10]:
# Ensure we have data even if no refinement happened (fallback)
if not s2_summary: 
    # Try to find last state from history
    pass 

if not s2_summary and 'summary' in locals().get('final_state_values', {}):
    s2_summary = final_state_values['summary']
if not s2_actions and 'action_items' in locals().get('final_state_values', {}):
    s2_actions = final_state_values['action_items']

from IPython.display import display, HTML
import html as html_lib

# --- Helper to format Action Items Table ---
def create_action_items_table(actions):
    if not actions: return "<em>No action items</em>"
    
    rows = ""
    for idx, item in enumerate(actions, 1):
        rows += f"""
        <tr>
            <td><b>{idx}</b></td>
            <td style="text-align:left"><b>{item.get('title', '')}</b><br><span style="color:#666; font-size:0.9em">{item.get('description', '') or ''}</span></td>
            <td>{item.get('assignee', '')}</td>
            <td>{item.get('priority', '')}</td>
            <td>{item.get('status', '')}</td>
            <td>{item.get('due_date', '')}</td>
        </tr>
        """
    
    return f"""
    <table class="t">
        <thead>
            <tr>
                <th style="width:5%">#</th>
                <th style="width:40%">Task & Description</th>
                <th style="width:15%">Assignee</th>
                <th style="width:10%">Priority</th>
                <th style="width:15%">Status</th>
                <th style="width:15%">Due Date</th>
            </tr>
        </thead>
        <tbody>{rows}</tbody>
    </table>
    """

def safe_html(text):
    if not text: return ""
    return html_lib.escape(text).replace("\n", "<br>")

# --- Construct The View ---

html_content = f"""
<style>
    .t {{ font-family: Arial, sans-serif; border-collapse: collapse; width: 100%; margin-bottom: 20px; font-size: 14px; }}
    .t th {{ background: #f3f4f6; color: #111; padding: 10px; text-align: left; border: 1px solid #e5e7eb; }}
    .t td {{ padding: 10px; border: 1px solid #e5e7eb; vertical-align: top; }}
    .h-title {{ color: #1f2937; margin-top: 30px; border-bottom: 2px solid #3b82f6; padding-bottom: 5px; display: inline-block; }}
    .comparison-container {{ display: flex; gap: 20px; }}
    .col-half {{ flex: 1; }}
    .badge {{ display: inline-block; padding: 2px 8px; border-radius: 12px; font-size: 0.8em; font-weight: bold; background: #e0f2fe; color: #0284c7; }}
</style>

<h2 class="h-title">📊 Experiment Results Comparison</h2>

<h3>1. Summary Comparison</h3>
<div class="comparison-container">
    <div class="col-half">
        <h4>🚫 Scenario 1: No Reflection</h4>
        <div style="padding:15px; border-radius:8px; border:1px solid #e5e7eb; height: 100%;">
            {safe_html(s1_summary)}
        </div>
    </div>
    <div class="col-half">
        <h4>✅ Scenario 2: With Reflection</h4>
        <div style="padding:15px; border-radius:8px; border:1px solid #bbf7d0; height: 100%;">
            {safe_html(s2_summary)}
        </div>
    </div>
</div>

<h3>2. Action Items Analysis</h3>

<h4>Scenario 1: No Reflection ({len(s1_actions)} items)</h4>
{create_action_items_table(s1_actions)}

<h4>Scenario 2: With Reflection ({len(s2_actions)} items)</h4>
{create_action_items_table(s2_actions)}

<h3>3. Agent self-correction</h3>
<p>Examples of critiques that led to changes in Scenario 2:</p>
<ul>
    {''.join(f"<li>{c}</li>" for c in critiques_log)}
</ul>

"""

display(HTML(html_content))

#,Task & Description,Assignee,Priority,Status,Due Date
1,Hoàn thành module AuthenticationTích hợp Google và Github login,Nam,High,To Do,2024-01-19
2,Thiết kế DashboardHoàn thành giao diện Dashboard,Lan,High,To Do,2024-01-19
3,Cập nhật API KeyUpdate API Key mới trong file .env,Unassigned,Urgent,To Do,2024-01-08
#,Task & Description,Assignee,Priority,Status,Due Date
1,Hoàn thành module AuthenticationTích hợp Google và Github login,Nam,High,To Do,2024-01-19
2,Review code AuthenticationReview code Auth cho Nam,Tuấn,High,To Do,2024-01-19
3,Thiết kế DashboardHoàn thành giao diện Dashboard,Lan,High,To Do,2024-01-19
4,Cập nhật API KeyUpdate API Key mới trong file .env,Hùng,Urgent,To Do,2024-01-08


## Detailed Results Comparison
Below are the comprehensive results for each scenario, including the full summary and all action items generated.